# 10. Predict, check and explain

**Open:** restart R; save the supplied script as `scripts/my-10-test-tree.R`.

**Input:** Chapter 9's saved tree. **Question:** did the extra complexity earn its place?

## 10.1 Predict without learning from the answers

Run the supplied `load-tree-fit` block:

In [1]:
source("scripts/tree_helpers.R")
require_tree_package()
if (!file.exists("outputs/checkpoints/tree-fit.rds")) stop("Run Chapter 9 first to create tree-fit.rds.")
chosen <- readRDS("outputs/checkpoints/tree-fit.rds")
final_tree <- chosen$fit
train_filled <- fill_predictors(chosen$train, chosen$medians)

Then run `predict-tree`. The test inputs receive the **saved training medians**. `predict()` applies the fitted tree; it does not refit it:

In [2]:
# Test features use medians already learned from training. No refit or new choice.
test_filled <- fill_predictors(chosen$test, chosen$medians)
tree_prediction <- as.numeric(predict(final_tree, newdata = test_filled))
baseline_prediction <- rep(mean(chosen$train$log10_msc), nrow(test_filled))
head(data.frame(observed_log10 = test_filled$log10_msc,
                tree_prediction = tree_prediction, baseline_prediction = baseline_prediction), 3)

,observed_log10,tree_prediction,baseline_prediction
,<dbl>,<dbl>,<dbl>
1,0.8129134,0.6873386,0.7450425
2,0.5440680,0.6873386,0.7450425
3,-0.3010300,0.6873386,0.7450425


The baseline still uses the same training mean as Chapter 7. Both methods predict the same 105 records from 21 held-out farms, on the same log10 target scale.

## 10.2 Let the comparison answer the question

**Before running:** what evidence would justify keeping the more complicated rule?

Run `tree-test-scores`:

In [3]:
tree_scores <- data.frame(model = c("Training-mean baseline", "Shallow regression tree"),
  maxdepth = c(0, chosen$chosen_depth), n_test = nrow(test_filled), test_farms = length(unique(test_filled$FarmID)),
  RMSE_log10_CFU_ml = c(sqrt(mean((test_filled$log10_msc - baseline_prediction)^2)),
                       sqrt(mean((test_filled$log10_msc - tree_prediction)^2))),
  MAE_log10_CFU_ml = c(mean(abs(test_filled$log10_msc - baseline_prediction)),
                      mean(abs(test_filled$log10_msc - tree_prediction))))
tree_scores

model,maxdepth,n_test,test_farms,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>
Training-mean baseline,0,105,21,0.5289231,0.4091480
Shallow regression tree,1,105,21,0.5448970,0.4264264


In this supplied example the tree's RMSE is about **0.545**, compared with **0.529** for the baseline. Its MAE is also higher. The added model did **not** improve these test predictions. The baseline was also slightly better in the training-only OOF comparison.

That is a result you can explain. Two survey variables and a shallow tree did not earn extra complexity here; it does not show that all trees fail or that the inputs have no biological relevance. Keep this result rather than repeatedly changing the analysis to improve a now-visible test score. The test split was already displayed in the baseline lesson, so this remains a fixed teaching comparison, not a fresh blinded evaluation.

## 10.3 Check what a leaf actually predicts

Run the supplied `check-leaves` block:

In [4]:
nodes <- extract_tree_nodes(final_tree)
train_routes <- lapply(seq_len(nrow(train_filled)), function(i) trace_tree(train_filled[i, ], nodes))
train_leaf <- vapply(train_routes, function(route) route$leaf, integer(1))
train_prediction <- as.numeric(predict(final_tree, newdata = train_filled))
stopifnot(isTRUE(all.equal(vapply(train_routes, function(route) route$prediction, numeric(1)), train_prediction)))
leaf_means <- aggregate(train_filled$log10_msc, list(leaf = train_leaf), mean)
names(leaf_means)[2] <- "mean_from_training_records"
leaf_means$tree_prediction <- nodes$mean_log10[match(leaf_means$leaf, nodes$node)]
stopifnot(isTRUE(all.equal(leaf_means$mean_from_training_records, leaf_means$tree_prediction)))
leaf_means

leaf,mean_from_training_records,tree_prediction
<int>,<dbl>,<dbl>
2,0.6873386,0.6873386
3,1.0545454,1.0545454


One column recomputes the average outcome from the training records in each leaf; the other reads the fitted tree's prediction. They agree. This is the same averaging idea as the baseline, now applied separately to groups.

Run `trace-one` to follow the first test record. The table shows the recorded inputs and what the model used after any missing-value filling:

In [5]:
first_route <- trace_tree(test_filled[1, ], nodes)
data.frame(feature = tree_features, recorded = unlist(chosen$test[1, tree_features], use.names = FALSE),
           used_by_model = unlist(test_filled[1, tree_features], use.names = FALSE))
first_route
stopifnot(isTRUE(all.equal(first_route$prediction, tree_prediction[1])))

feature,recorded,used_by_model
<chr>,<dbl>,<dbl>
pasture_hours,12,12
milking_cows,50,50


$leaf
[1] 2

$prediction
[1] 0.6873386

$path
[1] "milking_cows >= 36.5"

The supplied tree splits at **36.5 milking cows**. There is no half cow: this separates integer counts of 36 or fewer from 37 or more. The first record has 50 cows, enters leaf 2 and receives its training mean, about 0.6873 log10 MSC.

Pasture hours were offered to the model but were not used in this selected tree. A learned branch is an association in this training data, not advice to change herd size or a safety threshold.

## 10.4 Save, reload and use the model

Run the supplied `save-tree-results` block:

In [6]:
write.csv(tree_scores, "outputs/tree/test-scores.csv", row.names = FALSE)
write.csv(nodes, "outputs/tree/nodes.csv", row.names = FALSE)
write.csv(leaf_means, "outputs/tree/leaf-means.csv", row.names = FALSE)
write.csv(data.frame(feature = tree_features, median = unname(chosen$medians)),
          "outputs/tree/training-medians.csv", row.names = FALSE)
training_predictions <- data.frame(FarmID = train_filled$FarmID, Sampling = train_filled$Sampling,
  observed_log10 = train_filled$log10_msc, train_filled[tree_features], leaf = train_leaf,
  tree_prediction = train_prediction)
write.csv(training_predictions, "outputs/tree/training-predictions.csv", row.names = FALSE)
test_predictions <- data.frame(FarmID = test_filled$FarmID, Sampling = test_filled$Sampling,
  observed_log10 = test_filled$log10_msc, test_filled[tree_features],
  pasture_hours_missing = is.na(chosen$test$pasture_hours), milking_cows_missing = is.na(chosen$test$milking_cows),
  tree_prediction = tree_prediction, baseline_prediction = baseline_prediction)
write.csv(test_predictions, "outputs/tree/test-predictions.csv", row.names = FALSE)
saveRDS(list(fit = final_tree, medians = chosen$medians, features = tree_features), "outputs/tree/model.rds")
writeLines(capture.output(sessionInfo()), "outputs/tree/session-info.txt")

`outputs/tree/model.rds` stores the fitted tree and training medians together. Keeping both lets new inputs receive the same preparation used by the model.

**Predict first:** the next two input profiles both have 6 pasture hours, but one has 50 cows and the other 30. Which leaf will each enter? These profiles are invented and have no observed MSC, so they cannot be used to calculate predictive accuracy.

Run `new-samples` to reload the saved model and predict:

In [7]:
# Invented input profiles, without observed outcomes; not an evaluation dataset.
saved_model <- readRDS("outputs/tree/model.rds")
new_samples <- data.frame(pasture_hours = c(6, 6), milking_cows = c(50, 30))
new_ready <- fill_predictors(new_samples, saved_model$medians)
new_prediction <- as.numeric(predict(saved_model$fit, newdata = new_ready))
example_predictions <- data.frame(new_samples, prediction_log10 = new_prediction,
                                  concentration_expression_CFU_ml = 10^new_prediction)
example_predictions
write.csv(example_predictions, "outputs/tree/example-predictions.csv", row.names = FALSE)

pasture_hours,milking_cows,prediction_log10,concentration_expression_CFU_ml
<dbl>,<dbl>,<dbl>,<dbl>
6,50,0.6873386,4.867866
6,30,1.0545454,11.338235


The final column expresses each log10 prediction as `10^prediction` CFU/mL, as in Chapter 7. It does not turn the prediction into an arithmetic-mean concentration or a safety verdict. Comparing these two profiles is not an intervention experiment.

In your **my- copy**, change one input and predict the direction or lack of change before running again. Explain the answer using the branch, rather than asking AI to invent a biological story.

## 10.5 Inspect the tree and keep your evidence

Run the supplied `plot-tree` block, then open the saved image:

In [8]:
write_tree_mermaid(nodes, "outputs/tree/fitted-tree.mmd")
png("outputs/tree/fitted-tree.png", width = 1200, height = 720, res = 150)
par(mar = c(0, 0, 3, 0), xpd = NA, family = "sans")
draw_teaching_tree(final_tree, nodes)
invisible(dev.off())
cat("Saved outputs/tree/fitted-tree.png\n")

Saved outputs/tree/fitted-tree.png


![Fitted shallow regression tree. Leaves show mean log10 MSC and the number of training records.](../outputs/tree/fitted-tree.png)

| Find in outputs/tree | What it lets you explain |
|---|---|
| `predictor-audit.csv` | Missing survey information and records kept |
| `validation-scores.csv` | Why this depth was selected |
| `test-scores.csv` | Same-record comparison with the baseline |
| `leaf-means.csv` and `nodes.csv` | Where the predictions came from |
| `model.rds` and `example-predictions.csv` | Reuse of the fitted rule and its preparation |

**Write three sentences:** name the inputs and validation choice; compare the two test errors; state one limit. Missing survey answers were filled by a simple rule, samples repeat within farms, and this split cannot establish performance in a different region, causal effects or milk safety.

## 10.6 Take the workflow to another model

| Step | When the algorithm changes |
|---|---|
| Define the task | Keep the target, available information and intended new-farm setting explicit |
| Prepare inputs | A new method may need different preparation; learn it inside each fitting partition |
| Fit and choose settings | Replace the tree and depth setting; keep whole-farm training validation |
| Predict and compare | Evaluate the chosen procedure on appropriate held-out records, without using their answers to choose it |
| Explain and verify | Keep saved predictions and checks; leaf means are specific to this tree, so other models need their own explanation checks |

**Finish:** save optional changed-input results under a different filename if you want to keep them. Restart R and run a my- copy of `scripts/full-analysis.R`; it recreates the supplied reference baseline and tree workflow from the original source downloads. Explain which parts of the table you could reuse for your next model.